In [ ]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
import os
import random
from tqdm import tqdm
from scipy import sparse
from scipy.sparse.linalg import spsolve

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Device setup
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(7)

In [ ]:
# BASELINE CORRECTION (Asymmetric Least Squares)
# ALS estimates the fluorescence background and subtracts it 

def als_baseline(y, lam=1e5, p=0.001, niter=10):
    L = len(y)
    D = sparse.diags([1, -2, 1], [0, -1, -2], shape=(L, L-2))
    D = lam * D.dot(D.transpose())
    w = np.ones(L)
    for _ in range(niter):
        W = sparse.diags(w, 0)
        Z = W + D
        z = spsolve(Z, w * y)
        w = p * (y > z) + (1 - p) * (y < z)
    return z


def correct_baseline(profiles):
    corrected = np.zeros_like(profiles)
    for i, profile in enumerate(profiles):
        baseline = als_baseline(profile)
        corrected[i] = profile - baseline
        # Clip negatives to zero (can't have negative intensity)
        corrected[i] = np.clip(corrected[i], 0, None)
    return corrected


print('Baseline correction function ready.')
print('ALS = Asymmetric Least Squares — estimates fluorescence background and subtracts it.')

In [ ]:
# LOAD DATA FROM CSV 
BASE_PATH = os.path.expanduser('~/Downloads/rock_csvs')

data_183 = {
    'S10Granite':           os.path.join(BASE_PATH, 'S10Granite_1-83Hz_profiles.csv'),
    'Holstein_Sandstone':   os.path.join(BASE_PATH, 'Holstein_Sandstone_1-83Hz_profiles.csv'),
    'Leitendorf_Limestone': os.path.join(BASE_PATH, 'Leitendorf_Limestone_1-83Hz_profiles.csv'),
}

data_510 = {
    'S10Granite':           os.path.join(BASE_PATH, 'S10Granite_5-10Hz_profiles.csv'),
    'Holstein_Sandstone':   os.path.join(BASE_PATH, 'Holstein_Sandstone_5-10Hz_profiles.csv'),
    'Leitendorf_Limestone': os.path.join(BASE_PATH, 'Leitendorf_Limestone_5-10Hz_profiles.csv'),
}


def load_data(data_dict):
    dfs = []
    class_names = list(data_dict.keys())
    for label, (rock_name, path) in enumerate(data_dict.items()):
        df = pd.read_csv(path, header=None, decimal='.')
        df_numeric = df.iloc[3::6, :1060].astype(np.float32)
        class_num = np.array([label] * df_numeric.shape[0], dtype=np.float32)
        d = np.c_[df_numeric.values, class_num]
        dfs.extend(d)
        print(f'{rock_name}: {df_numeric.shape[0]} profiles')
    return np.array(dfs), class_names


print('Loading 1.83 Hz...')
dataset_183, class_names = load_data(data_183)
print(f'Shape: {dataset_183.shape}\n')

print('Loading 5.10 Hz...')
dataset_510, _ = load_data(data_510)
print(f'Shape: {dataset_510.shape}')

In [ ]:
# APPLY BASELINE CORRECTION TO GRANITE ONLY

def apply_granite_baseline_correction(dataset, class_names):
    granite_label = class_names.index('S10Granite')
    X = dataset[:, :-1].copy()
    y = dataset[:, -1:]

    granite_mask = (y.flatten() == granite_label)
    print(f'Applying baseline correction to {granite_mask.sum()} granite profiles...')

    X[granite_mask] = correct_baseline(X[granite_mask])
    print('Done.')
    return np.c_[X, y]


print('Correcting 1.83 Hz granite...')
dataset_183_corrected = apply_granite_baseline_correction(dataset_183, class_names)

print('\nCorrecting 5.10 Hz granite...')
dataset_510_corrected = apply_granite_baseline_correction(dataset_510, class_names)

In [ ]:
# VISUALIZE BEFORE VS AFTER BASELINE CORRECTION 
X_183 = dataset_183[:, :-1]
y_183 = dataset_183[:, -1:]
X_183_corr = dataset_183_corrected[:, :-1]

granite_idx = np.where(y_183.flatten() == 0)[0][:5]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for j in granite_idx:
    axes[0].plot(X_183[j], alpha=0.6)
axes[0].set_title('Granite - BEFORE baseline correction')
axes[0].set_xlabel('Wavelength index')
axes[0].set_ylabel('Intensity')

for j in granite_idx:
    axes[1].plot(X_183_corr[j], alpha=0.6)
axes[1].set_title('Granite - AFTER baseline correction')
axes[1].set_xlabel('Wavelength index')
axes[1].set_ylabel('Intensity')

plt.suptitle('ALS Baseline Correction Effect on Granite', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# DATASET PREPARATION 
X_183_corr = dataset_183_corrected[:, :-1]
y_183      = dataset_183_corrected[:, -1:]
X_510_corr = dataset_510_corrected[:, :-1]
y_510      = dataset_510_corrected[:, -1:]

X_train_183, X_test_183, y_train_183, y_test_183 = train_test_split(
    X_183_corr, y_183, test_size=0.2, random_state=3, stratify=y_183
)
X_train_510, X_test_510, y_train_510, y_test_510 = train_test_split(
    X_510_corr, y_510, test_size=0.2, random_state=3, stratify=y_510
)

print('1.83 Hz -> Train:', X_train_183.shape, ' Test:', X_test_183.shape)
print('5.10 Hz -> Train:', X_train_510.shape, ' Test:', X_test_510.shape)

In [ ]:
class RockDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    def __len__(self):
        return len(self.X)


train_dataset_183 = RockDataset(X_train_183, y_train_183)
test_dataset_183  = RockDataset(X_test_183,  y_test_183)
train_dataset_510 = RockDataset(X_train_510, y_train_510)
test_dataset_510  = RockDataset(X_test_510,  y_test_510)

In [ ]:
import itertools

batch_sizes   = [2048]
lrs           = [0.0001, 0.00001]
epochs_list   = [100]
weight_decays = [0., 1e-4]

hyperparameters_list = []
for batch_size, lr, epochs, weight_decay in itertools.product(batch_sizes, lrs, epochs_list, weight_decays):
    hyperparameters_list.append({'batch_size': batch_size, 'lr': lr, 'epochs': epochs, 'weight_decay': weight_decay})

print(f'Generated {len(hyperparameters_list)} combinations.')

In [ ]:
# MODEL (3 conv layers) 
class Model(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.layer_norm_input = nn.LayerNorm(normalized_shape=1060)
        self.conv1 = nn.Conv1d(1,  16, kernel_size=9, stride=3)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=9, stride=3)
        self.conv3 = nn.Conv1d(32, 64, kernel_size=5, stride=2)
        self.layer_norm_1 = nn.LayerNorm(normalized_shape=64)
        self.fnn1 = nn.Linear(64, n_classes)
        self.relu        = nn.ReLU()
        self.max_pool    = nn.MaxPool1d(kernel_size=2)
        self.global_pool = nn.AdaptiveMaxPool1d(output_size=1)
        self.flatten     = nn.Flatten()
        self.dropout     = nn.Dropout(p=0.2)

    def forward(self, x):
        x = self.layer_norm_input(x)
        x = x.unsqueeze(1)
        x = self.relu(self.conv1(x))
        x = self.max_pool(x)
        x = self.relu(self.conv2(x))
        x = self.max_pool(x)
        x = self.relu(self.conv3(x))
        x = self.global_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.layer_norm_1(x)
        x = self.fnn1(x)
        return x

print(f'3-class model parameters: {sum(p.numel() for p in Model(3).parameters())}')
print(f'2-class model parameters: {sum(p.numel() for p in Model(2).parameters())}')

In [ ]:
# TRAINING FUNCTION 
def run_experiments(train_dataset, test_dataset, X_test, y_test, speed_tag, results_dir, n_classes, names):

    experiment_results = []
    os.makedirs(results_dir, exist_ok=True)

    monitor_indices = np.arange(4)
    X_monitor = torch.from_numpy(X_test[monitor_indices]).to(device)
    y_monitor = torch.from_numpy(y_test[monitor_indices]).squeeze().long().to(device)
    print(f'Monitor True Labels: {[names[int(i)] for i in y_monitor]}')

    for idx, config in enumerate(hyperparameters_list):
        print(f"\n{'='*20}\nExperiment {idx+1}\nConfig: {config}\n{'='*20}")

        train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True,  num_workers=0)
        test_loader  = DataLoader(test_dataset,  batch_size=config['batch_size'], shuffle=False, num_workers=0)

        model     = Model(n_classes).to(device)
        optimizer = torch.optim.Adam(params=model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
        criterion = nn.CrossEntropyLoss()

        best_accuracy   = 0.0
        best_model_path = os.path.join(results_dir, f'best_model_exp_{idx+1}.pth')
        writer          = SummaryWriter(log_dir=os.path.join(results_dir, f'tb_exp{idx+1}'))

        training_loss, training_acc, testing_loss, testing_acc = [], [], [], []

        for epoch in range(config['epochs']):
            print(f'Epoch: {epoch+1}/{config["epochs"]}')

            model.train()
            avg_train_loss, avg_train_acc = [], []
            for i, d in tqdm(enumerate(train_loader), total=len(train_loader), leave=False):
                X_batch, y_batch = d
                X_batch = X_batch.to(device)
                y_batch = y_batch.squeeze().long().to(device)
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss    = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                _, predicted = torch.max(outputs, 1)
                avg_train_loss.append(loss.item())
                avg_train_acc.append((predicted == y_batch).sum().item() / y_batch.size(0))

            avg_acc_train  = sum(avg_train_acc)  / len(avg_train_acc)
            avg_loss_train = sum(avg_train_loss) / len(avg_train_loss)
            training_acc.append(avg_acc_train)
            training_loss.append(avg_loss_train)
            print(f'Training: Loss={avg_loss_train:.4f}, Acc={avg_acc_train:.4f}')

            model.eval()
            avg_test_loss, avg_test_acc = [], []
            with torch.no_grad():
                for i, d in tqdm(enumerate(test_loader), total=len(test_loader), leave=False):
                    X_batch, y_batch = d
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.squeeze().long().to(device)
                    outputs = model(X_batch)
                    loss    = criterion(outputs, y_batch)
                    preds   = torch.max(outputs, 1)[1]
                    avg_test_loss.append(loss.item())
                    avg_test_acc.append((preds == y_batch).sum().item() / y_batch.size(0))

            avg_acc_test  = sum(avg_test_acc)  / len(avg_test_acc)
            avg_loss_test = sum(avg_test_loss) / len(avg_test_loss)
            testing_acc.append(avg_acc_test)
            testing_loss.append(avg_loss_test)
            print(f'Testing:  Loss={avg_loss_test:.4f}, Acc={avg_acc_test:.4f}\n')

            writer.add_scalars('Loss',     {'train': avg_loss_train, 'test': avg_loss_test}, epoch)
            writer.add_scalars('Accuracy', {'train': avg_acc_train,  'test': avg_acc_test},  epoch)

            if avg_acc_test > best_accuracy:
                best_accuracy = avg_acc_test
                torch.save(model.state_dict(), best_model_path)
                print(f' New Best Acc: {best_accuracy:.4f}')

            model.eval()
            with torch.no_grad():
                outputs_monitor = model(X_monitor)
                _, pred_monitor = torch.max(outputs_monitor, 1)
                print(f'Monitor (Epoch {epoch+1}):')
                for j, (true_cls, pred_cls) in enumerate(zip(y_monitor, pred_monitor)):
                    status = '\u2713' if true_cls == pred_cls else '\u2717'
                    print(f'  {names[true_cls.item()]:<20} | Pred: {names[pred_cls.item()]:<20} {status}')
            print('-' * 30)

        # Confusion Matrix
        model.load_state_dict(torch.load(best_model_path, map_location=device))
        model.to(device).eval()
        all_preds, all_true = [], []
        with torch.no_grad():
            for i, d in tqdm(enumerate(test_loader), total=len(test_loader), leave=False):
                X_batch, y_batch = d
                X_batch = X_batch.to(device)
                y_batch = y_batch.squeeze().long().to(device)
                preds = torch.max(model(X_batch), 1)[1]
                all_preds.extend(preds.cpu().numpy())
                all_true.extend(y_batch.cpu().numpy())

        cm = confusion_matrix(all_true, all_preds)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=names, yticklabels=names)
        plt.title(f'Confusion Matrix - {speed_tag} - Exp {idx+1}')
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.tight_layout()
        cm_path = os.path.join(results_dir, f'confusion_matrix_exp_{idx+1}.png')
        plt.savefig(cm_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'[SAVED] Confusion matrix -> {cm_path}')

        writer.close()
        experiment_results.append({
            'config': config, 'final_test_acc': testing_acc[-1],
            'final_test_loss': testing_loss[-1], 'best_model_path': best_model_path,
            'history': {'train_loss': training_loss, 'train_acc': training_acc,
                        'test_loss': testing_loss, 'test_acc': testing_acc}
        })

    print('\n' + '='*40)
    print(f'RESULTS - {speed_tag}')
    print('='*40)
    for res in experiment_results:
        print(f"Config: {res['config']} -> Acc: {res['final_test_acc']:.4f}")

    return experiment_results

In [ ]:
# PART 1: 3-CLASS WITH BASELINE CORRECTED GRANITE

print('\n' + '#'*50)
print('3-CLASS TRAINING WITH BASELINE CORRECTION - 1.83 Hz')
print('#'*50)
results_3class_183 = run_experiments(
    train_dataset_183, test_dataset_183,
    X_test_183, y_test_183,
    '1.83Hz_corrected',
    'results_1d_cnn_als_correction_and_binary_cl/corrected_183',
    n_classes=3,
    names=class_names
)

print('\n' + '#'*50)
print('3-CLASS TRAINING WITH BASELINE CORRECTION - 5.10 Hz')
print('#'*50)
results_3class_510 = run_experiments(
    train_dataset_510, test_dataset_510,
    X_test_510, y_test_510,
    '5.10Hz_corrected',
    'results_1d_cnn_als_correction_and_binary_cl/corrected_510',
    n_classes=3,
    names=class_names
)

In [ ]:
# COMPARE BEFORE vs AFTER BASELINE CORRECTION 
best_183_corr = max(results_3class_183, key=lambda x: x['final_test_acc'])
best_510_corr = max(results_3class_510, key=lambda x: x['final_test_acc'])

print('='*50)
print('COMPARISON: Before vs After Baseline Correction')
print('='*50)
print(f'1.83Hz - Before: ~90%  |  After: {best_183_corr["final_test_acc"]*100:.1f}%')
print(f'5.10Hz - Before: ~90%  |  After: {best_510_corr["final_test_acc"]*100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(best_183_corr['history']['train_acc'], color='blue', label='Train')
axes[0].plot(best_183_corr['history']['test_acc'],  color='red',  label='Test')
axes[0].set_title('Accuracy - 1.83Hz (Baseline Corrected)')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(best_510_corr['history']['train_acc'], color='blue', label='Train')
axes[1].plot(best_510_corr['history']['test_acc'],  color='red',  label='Test')
axes[1].set_title('Accuracy - 5.10Hz (Baseline Corrected)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
RES = 'results_1d_cnn_als_correction_and_binary_cl'
os.makedirs(RES, exist_ok=True)
_path = os.path.join(RES, 'accuracy_curves_corrected.png')
plt.savefig(_path, dpi=150, bbox_inches='tight')
print(f'[SAVED] Corrected accuracy curves -> {_path}')
plt.show()

In [ ]:
# PART 2: BINARY CLASSIFIER - SANDSTONE vs LIMESTONE ONLY
 
print('Building binary dataset (Sandstone vs Limestone)...')

def build_binary_dataset(dataset, class_names):
    sandstone_label = class_names.index('Holstein_Sandstone')
    limestone_label = class_names.index('Leitendorf_Limestone')

    X = dataset[:, :-1]
    y = dataset[:, -1:].flatten()

    # Keep only sandstone and limestone
    mask = (y == sandstone_label) | (y == limestone_label)
    X_bin = X[mask]
    y_bin = y[mask]

    # Relabel: sandstone=0, limestone=1
    y_bin = (y_bin == limestone_label).astype(np.float32).reshape(-1, 1)
    binary_names = ['Holstein_Sandstone', 'Leitendorf_Limestone']

    print(f'  Sandstone: {(y_bin==0).sum()} profiles')
    print(f'  Limestone: {(y_bin==1).sum()} profiles')
    return X_bin, y_bin, binary_names


print('1.83 Hz binary:')
X_bin_183, y_bin_183, binary_names = build_binary_dataset(dataset_183_corrected, class_names)
print('5.10 Hz binary:')
X_bin_510, y_bin_510, _            = build_binary_dataset(dataset_510_corrected, class_names)

In [ ]:
# VISUALIZE SANDSTONE vs LIMESTONE 
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sandstone_idx = np.where(y_bin_183.flatten() == 0)[0][:5]
limestone_idx = np.where(y_bin_183.flatten() == 1)[0][:5]

for j in sandstone_idx:
    axes[0].plot(X_bin_183[j], alpha=0.6, color='blue')
axes[0].set_title('Holstein Sandstone - 1.83 Hz')
axes[0].set_xlabel('Wavelength index')
axes[0].set_ylabel('Intensity')

for j in limestone_idx:
    axes[1].plot(X_bin_183[j], alpha=0.6, color='orange')
axes[1].set_title('Leitendorf Limestone - 1.83 Hz')
axes[1].set_xlabel('Wavelength index')
axes[1].set_ylabel('Intensity')

plt.suptitle('Sandstone vs Limestone Profiles', fontsize=14)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(X_bin_183[y_bin_183.flatten()==0].mean(axis=0), color='blue',   label='Sandstone (mean)', linewidth=2)
plt.plot(X_bin_183[y_bin_183.flatten()==1].mean(axis=0), color='orange', label='Limestone (mean)',  linewidth=2)
plt.title('Mean Raman Profile: Sandstone vs Limestone - 1.83 Hz')
plt.xlabel('Wavelength index')
plt.ylabel('Intensity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# TRAIN BINARY CLASSIFIER 
X_train_bin_183, X_test_bin_183, y_train_bin_183, y_test_bin_183 = train_test_split(
    X_bin_183, y_bin_183, test_size=0.2, random_state=3, stratify=y_bin_183
)
X_train_bin_510, X_test_bin_510, y_train_bin_510, y_test_bin_510 = train_test_split(
    X_bin_510, y_bin_510, test_size=0.2, random_state=3, stratify=y_bin_510
)

train_dataset_bin_183 = RockDataset(X_train_bin_183, y_train_bin_183)
test_dataset_bin_183  = RockDataset(X_test_bin_183,  y_test_bin_183)
train_dataset_bin_510 = RockDataset(X_train_bin_510, y_train_bin_510)
test_dataset_bin_510  = RockDataset(X_test_bin_510,  y_test_bin_510)

print('Binary train/test split done.')
print(f'1.83Hz — Train: {X_train_bin_183.shape}, Test: {X_test_bin_183.shape}')
print(f'5.10Hz — Train: {X_train_bin_510.shape}, Test: {X_test_bin_510.shape}')

In [ ]:
print('\n' + '#'*50)
print('BINARY CLASSIFIER: Sandstone vs Limestone - 1.83 Hz')
print('#'*50)
results_binary_183 = run_experiments(
    train_dataset_bin_183, test_dataset_bin_183,
    X_test_bin_183, y_test_bin_183,
    'Binary_1.83Hz',
    'results_1d_cnn_als_correction_and_binary_cl/binary_183',
    n_classes=2,
    names=binary_names
)

print('\n' + '#'*50)
print('BINARY CLASSIFIER: Sandstone vs Limestone - 5.10 Hz')
print('#'*50)
results_binary_510 = run_experiments(
    train_dataset_bin_510, test_dataset_bin_510,
    X_test_bin_510, y_test_bin_510,
    'Binary_5.10Hz',
    'results_1d_cnn_als_correction_and_binary_cl/binary_510',
    n_classes=2,
    names=binary_names
)

In [ ]:
# BINARY RESULTS SUMMARY 
best_bin_183 = max(results_binary_183, key=lambda x: x['final_test_acc'])
best_bin_510 = max(results_binary_510, key=lambda x: x['final_test_acc'])

print('='*50)
print('FINAL RESULTS SUMMARY')
print('='*50)
print(f'3-class (corrected) 1.83Hz: {max(r["final_test_acc"] for r in results_3class_183)*100:.1f}%')
print(f'3-class (corrected) 5.10Hz: {max(r["final_test_acc"] for r in results_3class_510)*100:.1f}%')
print(f'Binary (Sand vs Lime) 1.83Hz: {best_bin_183["final_test_acc"]*100:.1f}%')
print(f'Binary (Sand vs Lime) 5.10Hz: {best_bin_510["final_test_acc"]*100:.1f}%')

# Plot binary accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(best_bin_183['history']['train_acc'], color='blue',   label='Train')
axes[0].plot(best_bin_183['history']['test_acc'],  color='orange', label='Test')
axes[0].set_title('Binary Accuracy - 1.83Hz (Sand vs Lime)')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(best_bin_510['history']['train_acc'], color='blue',   label='Train')
axes[1].plot(best_bin_510['history']['test_acc'],  color='orange', label='Test')
axes[1].set_title('Binary Accuracy - 5.10Hz (Sand vs Lime)')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
RES = 'results_1d_cnn_als_correction_and_binary_cl'
os.makedirs(RES, exist_ok=True)
_path = os.path.join(RES, 'accuracy_curves_binary.png')
plt.savefig(_path, dpi=150, bbox_inches='tight')
print(f'[SAVED] Binary accuracy curves -> {_path}')
plt.show()

In [ ]:
# BINARY CONFUSION MATRICES 
for tag, best, test_ds in [
    ('1.83Hz', best_bin_183, test_dataset_bin_183),
    ('5.10Hz', best_bin_510, test_dataset_bin_510)
]:
    y_true, y_pred = [], []
    loader = DataLoader(test_ds, batch_size=2048, shuffle=False, num_workers=0)
    model = Model(2).to(device)
    model.load_state_dict(torch.load(best['best_model_path'], map_location=device))
    model.eval()
    with torch.no_grad():
        for X_batch, y_batch in loader:
            outputs      = model(X_batch.to(device))
            _, predicted = torch.max(outputs, 1)
            y_true.extend(y_batch.squeeze().tolist())
            y_pred.extend(predicted.cpu().tolist())

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=binary_names)
    fig, ax = plt.subplots(figsize=(7, 7))
    disp.plot(ax=ax, cmap=plt.cm.Blues, xticks_rotation='vertical')
    plt.title(f'Binary Confusion Matrix - {tag} (Best Acc: {best["final_test_acc"]*100:.1f}%)')
    plt.show()